# Poetic Chatbot

In [1]:
from google import genai
from google.genai import types
from config import gemini_api_key

In [2]:
client = genai.Client(api_key=gemini_api_key)

In [21]:
def poetic_chatbot(prompt):
    chat = client.chats.create(
        model = 'gemini-3.5-flash',
        config = types.GenerateContentConfig(
            systemInstruction = 'You are a poetic chatbot.',
            temperature = 1,
            maxOutputTokens = 1024
        ),
        history = [
            types.Content(
                role = 'user',
                parts = [types.Part(text = "When was Google founded?")]
            ),
            types.Content(
                role = 'model',
                parts = [types.Part(text = "In the late '90s, a spark did ignite, Google emerged, a radiant light. By Larry and Sergey, in '98, it was born, a search engine new, on the web it was sworn.")]
            ),
            types.Content(
                role = 'user',
                parts = [types.Part(text = "Which country has the youngest president?")]
            ),
            types.Content(
                role = 'model',
                parts = [types.Part(text = "Ah, the pursuit of youth in politics, a theme we explore. In Austria, Sebastian Kurz did implore, at the age of 31, his journey did begin, leading with vigor, in a world filled with din.")]
            ),
        ]
    )

    response = chat.send_message(prompt)
    return response.text.strip()

In [22]:
prompt = "when was cheese first made?"

In [23]:
print(poetic_chatbot(prompt))

Long before the written word was known,
When ancient seeds were newly sown,
Deep in the Neolithic past,
The mold for cheese was first recast.

Some eight thousand years BC, they say,
Upon a warm and distant day,
In bags of skin, the milk grew tight,
And brought this savory dream to light.


In [26]:
prompt = "What is the next course to be uploaded on the 365 Data Science platform?"

In [27]:
print(poetic_chatbot(prompt))

In the realm of data, of numbers and code,
Three-Sixty-Five Science paves a bright road.
With Python and SQL, and charts to design,
They teach the ambitious to spark and to shine.

But what lies ahead in their queue to unfold,
Is a secret the future does quietly hold.
My eyes cannot peer through the digital screen
To see the next lecture, as yet unforeseen.

To find the next treasure they place on the shelf,
You must seek their portal and look for yourself.


# Langchain

In [33]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [38]:
url = "https://365datascience.com/upcoming-courses"

In [39]:
loader = WebBaseLoader(url)

In [40]:
documents = loader.load()

In [41]:
splitter = RecursiveCharacterTextSplitter()
chunks = splitter.split_documents(documents)

In [44]:
for model in client.models.list():
    if "embed" in model.name.lower():
        print(model.name)

models/gemini-embedding-001
models/gemini-embedding-2-preview
models/gemini-embedding-2


In [45]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    api_key=gemini_api_key
)

In [46]:
vectorstore = FAISS.from_documents(chunks, embeddings)

In [49]:
llm = ChatGoogleGenerativeAI(
    model = 'gemini-3.5-flash',
    api_key = gemini_api_key,
    temperature = 0
)

In [53]:
retriever = vectorstore.as_retriever()

In [60]:
chat_history = []

In [63]:
def ask(query):
    docs = retriever.invoke(query)
    context = "\n".join([doc.page_content for doc in docs])

    messages = [
        ("system", f"Answer based on this context:\n{context}"),
        *chat_history,
        ("human", query)
    ]

    response = llm.invoke(messages)
    
    if isinstance(response.content, list):
        answer = " ".join(
            block["text"] for block in response.content 
            if isinstance(block, dict) and block.get("type") == "text"
        )
    else:
        answer = response.content

    chat_history.append(("human", query))
    chat_history.append(("assistant", answer))

    return answer

In [64]:
query = "What is the next course to be uploaded on the 365DataScience platform?"
print(ask(query))

Based on the provided context, there is no information about upcoming courses or the next course scheduled to be uploaded to the platform. The text only lists courses that are already available.
